# OpenFold3 ddG Stand

Ноутбук для запуска panel-based ddG стенда: WT плюс 19 замен для каждой выбранной позиции.

Что умеет этот workflow:
- удобно задавать эксперимент через один основной блок настроек
- валидировать входные молекулы и позиции до запуска
- хранить состояние прогона в `state.sqlite`, чтобы можно было аккуратно возобновлять анализ
- собирать итоговый consensus ranking по всем ddG-методам
- экспортировать CSV/JSON сводки для дальнейшего разбора

Рекомендуемый порядок работы:
1. Выполни блок подготовки окружения.
2. При необходимости скорректируй пути и runtime-настройки.
3. Заполни основной блок эксперимента.
4. Запусти предпросмотр и проверь панели мутаций.
5. Запусти основной расчёт.
6. Посмотри итоговую таблицу и экспортированные файлы.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_ROOT.parent
HELPERS_DIR = NOTEBOOK_ROOT / 'helpers'
OPENFOLD_REPO_DIR = Path(os.environ.get('OPENFOLD_REPO_DIR', str(REPO_ROOT / 'openfold-3'))).resolve()
SAFE_TRITON_CACHE_DIR = Path(os.environ.get('TRITON_CACHE_DIR', '/tmp/openfold3_triton_cache')).resolve()

# Для ddG stand DeepSpeed не обязателен. По умолчанию отключаем ранний optional import,
# чтобы окружения без libaio не падали ещё до запуска самого OpenFold.
os.environ.setdefault('OPENFOLD_DISABLE_OPTIONAL_DEEPSPEED_IMPORT', '1')
os.environ.setdefault('TRITON_CACHE_DIR', str(SAFE_TRITON_CACHE_DIR))
os.environ.setdefault('DS_BUILD_AIO', '0')
os.environ.setdefault('AIO', '0')
SAFE_TRITON_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for path in (HELPERS_DIR, OPENFOLD_REPO_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from of_notebook_lib import RuntimeConfig, preview_molecules, validate_molecules
from of_notebook_lib.query_builders import build_single_query_payload, normalize_molecules
from openfold3.panel_stand import PanelDdgStandRunner, PanelStandConfig
from openfold3.panel_summary import summarize_panel_state_db, write_panel_summary_outputs

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)
CANONICAL_AA = 'ACDEFGHIKLMNPQRSTVWY'

def parse_positions(positions_text: str) -> tuple[int, ...]:
    values = []
    for token in positions_text.replace('\n', ',').split(','):
        token = token.strip()
        if not token:
            continue
        values.append(int(token))
    if not values:
        raise ValueError('Нужно указать хотя бы одну позицию')
    return tuple(values)

def find_chain_sequence(molecules: list[dict], mutable_chain_id: str) -> str:
    for molecule in normalize_molecules(molecules):
        if mutable_chain_id in molecule['chain_ids']:
            sequence = molecule.get('sequence')
            if not sequence:
                raise ValueError(f'У цепи {mutable_chain_id} нет sequence')
            return str(sequence)
    raise ValueError(f'Цепь {mutable_chain_id} не найдена во входных молекулах')

def build_panel_preview(target_id: str, molecules: list[dict], mutable_chain_id: str, positions: tuple[int, ...]) -> pd.DataFrame:
    sequence = find_chain_sequence(molecules, mutable_chain_id)
    rows = []
    for position in positions:
        if position < 1 or position > len(sequence):
            raise ValueError(f'Позиция {position} выходит за длину цепи {mutable_chain_id} ({len(sequence)})')
        wt_residue = sequence[position - 1]
        rows.append({
            'panel_id': f'{target_id}_{mutable_chain_id}_{wt_residue}{position}',
            'цепь': mutable_chain_id,
            'позиция': position,
            'WT остаток': wt_residue,
            'число мутантов': len(CANONICAL_AA) - 1,
        })
    return pd.DataFrame(rows)

def render_info_card(title: str, rows: list[tuple[str, object]], accent: str = "#0b7285") -> None:
    items = "".join(
        f"<tr><td style='padding:6px 10px;font-weight:600;white-space:nowrap;'>{label}</td><td style='padding:6px 10px;'>{value}</td></tr>"
        for label, value in rows
    )
    html = f"""
    <div style="border:1px solid #d0d7de;border-left:6px solid {accent};border-radius:10px;padding:12px 14px;margin:8px 0;background:#fbfdff;">
      <div style="font-size:18px;font-weight:700;margin-bottom:8px;">{title}</div>
      <table style="border-collapse:collapse;">{items}</table>
    </div>
    """
    display(HTML(html))


## 1. Настройка runtime

Меняй этот блок только если окружение OpenFold3, MSA-кэш или runner лежат в нестандартном месте.


In [ ]:
runtime = RuntimeConfig(
    project_dir=REPO_ROOT,
    openfold_repo_dir=OPENFOLD_REPO_DIR,
    openfold_prefix=Path(os.environ.get('OPENFOLD_PREFIX', str(REPO_ROOT / '.venv'))),
    results_dir=REPO_ROOT / 'results',
    msa_cache_dir=REPO_ROOT / 'msa_cache' / 'colabfold_msas',
    triton_cache_dir=SAFE_TRITON_CACHE_DIR,
    fixed_msa_tmp_dir=REPO_ROOT / '.runtime' / 'of3_colabfold_msas',
    use_fused_attention=False,
    use_deepspeed=False,
)

runtime_env = runtime.build_env()
for key, value in runtime_env.items():
    os.environ[key] = value
os.environ['OPENFOLD_PROJECT_DIR'] = str(runtime.project_dir)
os.environ['OPENFOLD_REPO_DIR'] = str(runtime.openfold_repo_dir)

render_info_card(
    "Runtime и безопасные env-переменные",
    [
        ("project_dir", runtime.project_dir),
        ("openfold_repo_dir", runtime.openfold_repo_dir),
        ("openfold_runner", runtime.openfold_runner),
        ("TRITON_CACHE_DIR", os.environ.get("TRITON_CACHE_DIR")),
        ("OPENFOLD_DISABLE_OPTIONAL_DEEPSPEED_IMPORT", os.environ.get("OPENFOLD_DISABLE_OPTIONAL_DEEPSPEED_IMPORT")),
        ("DS_BUILD_AIO", os.environ.get("DS_BUILD_AIO")),
    ],
    accent="#1c7ed6",
)


## 2. Диагностика окружения

Этот блок сразу показывает, нет ли типичных проблем с `run_openfold`, DeepSpeed и системной библиотекой `libaio`.


In [ ]:
deepspeed_status = "не установлен"
deepspeed_hint = "Для ddG stand это допустимо: notebook по умолчанию работает без optional DeepSpeed import."
try:
    import importlib.util
    if importlib.util.find_spec("deepspeed") is not None:
        deepspeed_status = "установлен"
        deepspeed_hint = "Если захочешь включать DeepSpeed-ускорение, на сервере должен быть установлен libaio-dev/libaio1-dev."
except Exception as exc:
    deepspeed_status = f"ошибка проверки: {exc}"

libaio_candidates = [
    Path("/usr/lib/x86_64-linux-gnu/libaio.so"),
    Path("/usr/lib64/libaio.so"),
    Path("/lib/x86_64-linux-gnu/libaio.so.1"),
]
libaio_found = any(path.exists() for path in libaio_candidates)
runner_found = Path(runtime.openfold_runner).exists()

render_info_card(
    "Проверка окружения",
    [
        ("run_openfold найден", runner_found),
        ("DeepSpeed", deepspeed_status),
        ("libaio найден", libaio_found),
        ("Подсказка", deepspeed_hint),
    ],
    accent="#2b8a3e" if runner_found else "#c92a2a",
)

if not libaio_found:
    print("Если позже понадобится настоящий DeepSpeed, обычно помогает установка: sudo apt-get install libaio-dev")


## 3. Основная настройка эксперимента

Ниже один основной блок, который обычно достаточно отредактировать для запуска.


In [ ]:
preset_name = 'balanced'
presets = {
    'fast_smoke': {
        'num_diffusion_samples': 1,
        'num_model_seeds': 1,
        'msa_panel_workers': 1,
        'analysis_workers': 2,
        'cleanup_intermediates': True,
    },
    'balanced': {
        'num_diffusion_samples': 2,
        'num_model_seeds': 1,
        'msa_panel_workers': 1,
        'analysis_workers': 4,
        'cleanup_intermediates': True,
    },
    'thorough': {
        'num_diffusion_samples': 4,
        'num_model_seeds': 2,
        'msa_panel_workers': 2,
        'analysis_workers': 6,
        'cleanup_intermediates': False,
    },
}

preset = presets[preset_name]

experiment_name = 'ddg_panel_demo'
target_id = 'demo_target'

molecules = [
    {
        'molecule_type': 'protein',
        'chain_ids': ['A'],
        'sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
    },
    {
        'molecule_type': 'protein',
        'chain_ids': ['B'],
        'sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
    },
]

mutable_chain_id = 'B'
positions_text = '10, 15'

runner_yaml = None
inference_ckpt_path = None
inference_ckpt_name = None
msa_computation_settings_yaml = None

num_diffusion_samples = preset['num_diffusion_samples']
num_model_seeds = preset['num_model_seeds']
msa_panel_workers = preset['msa_panel_workers']
analysis_workers = preset['analysis_workers']
cleanup_intermediates = preset['cleanup_intermediates']

output_parent = runtime.results_dir / 'panel_ddg_stand'
summary_export_dirname = 'summary_exports'


## 4. Предпросмотр входа

Этот блок проверяет молекулы, разрешает позиции и показывает, какие панели будут созданы.


In [ ]:
issues = validate_molecules(molecules)
if issues:
    for issue in issues:
        print("INPUT ISSUE:", issue)
else:
    print("Входные молекулы выглядят корректно.")

positions = parse_positions(positions_text)
sequence = find_chain_sequence(molecules, mutable_chain_id)
panel_preview_df = build_panel_preview(target_id, molecules, mutable_chain_id, positions)
planned_jobs = int(panel_preview_df["число мутантов"].sum())

render_info_card(
    "План запуска",
    [
        ("target_id", target_id),
        ("mutable_chain_id", mutable_chain_id),
        ("длина mutable chain", len(sequence)),
        ("позиции", list(positions)),
        ("число панелей", len(panel_preview_df)),
        ("число мутантных задач", planned_jobs),
        ("preset", preset_name),
    ],
    accent="#7048e8",
)

display(preview_molecules(molecules))
display(panel_preview_df)


## 5. Итоговый план запуска

Здесь видно, какие файлы и каталоги будут реально использованы при расчёте.


In [ ]:
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_root = output_parent / f'{experiment_name}_{run_stamp}'
wt_query_path = run_root / 'wt_query.json'
wt_query_payload = build_single_query_payload(f'{target_id}_WT', molecules)

resolved_plan = {
    'run_root': str(run_root),
    'wt_query_path': str(wt_query_path),
    'target_id': target_id,
    'mutable_chain_id': mutable_chain_id,
    'positions': list(positions),
    'runner_yaml': None if runner_yaml is None else str(Path(runner_yaml)),
    'inference_ckpt_path': None if inference_ckpt_path is None else str(Path(inference_ckpt_path)),
    'inference_ckpt_name': inference_ckpt_name,
    'msa_computation_settings_yaml': None if msa_computation_settings_yaml is None else str(Path(msa_computation_settings_yaml)),
    'num_diffusion_samples': num_diffusion_samples,
    'num_model_seeds': num_model_seeds,
    'msa_panel_workers': msa_panel_workers,
    'analysis_workers': analysis_workers,
    'cleanup_intermediates': cleanup_intermediates,
}

render_info_card(
    "Собранный план",
    [(key, value) for key, value in resolved_plan.items()],
    accent="#e67700",
)


## 6. Запуск эксперимента

Эта ячейка создаёт `wt_query.json`, запускает `PanelDdgStandRunner` и пишет summary exports.


In [ ]:
run_root.mkdir(parents=True, exist_ok=True)
wt_query_path.write_text(
    json.dumps(wt_query_payload, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

panel_config = PanelStandConfig(
    target_id=target_id,
    wt_query_json=wt_query_path,
    output_root=run_root,
    mutable_chain_id=mutable_chain_id,
    positions=positions,
    runner_yaml=None if runner_yaml is None else Path(runner_yaml),
    inference_ckpt_path=None if inference_ckpt_path is None else Path(inference_ckpt_path),
    inference_ckpt_name=inference_ckpt_name,
    msa_computation_settings_yaml=(None if msa_computation_settings_yaml is None else Path(msa_computation_settings_yaml)),
    num_diffusion_samples=num_diffusion_samples,
    num_model_seeds=num_model_seeds,
    msa_panel_workers=msa_panel_workers,
    analysis_workers=analysis_workers,
    cleanup_intermediates=cleanup_intermediates,
)

runner = PanelDdgStandRunner(panel_config)
try:
    run_payload = runner.run()
finally:
    runner.close()

panel_summary = summarize_panel_state_db(Path(run_payload['state_db']))
summary_outputs = write_panel_summary_outputs(
    panel_summary,
    Path(run_payload['output_root']) / summary_export_dirname,
)

panel_rows_df = pd.DataFrame([
    {
        'mutation': f'{row.chain_id}:{row.from_residue}{row.position_1based}{row.to_residue}',
        'predict_status': row.predict_status,
        'analysis_status': row.analysis_status,
        'consensus_z': row.consensus_z,
        'consensus_rank_desc': row.consensus_rank_desc,
        'rosetta_delta_vs_wt': row.rosetta_delta_vs_wt,
        **{f'score__{method}': row.method_scores.get(method) for method in panel_summary.methods},
    }
    for row in panel_summary.rows
])
top_consensus_df = pd.DataFrame(panel_summary.top_consensus)

render_info_card(
    "Эксперимент завершён",
    [
        ('run_root', run_payload['output_root']),
        ('state_db', run_payload['state_db']),
        ('summary_path', run_payload['summary_path']),
        ('panel_count', run_payload['panel_count']),
        ('mutant_job_count', run_payload['mutant_job_count']),
    ],
    accent="#2f9e44",
)
print(json.dumps({key: str(value) for key, value in summary_outputs.items()}, indent=2))


## 7. Результаты

Сначала выводится shortlist по consensus ranking, затем полная таблица по всем мутантам.


In [ ]:
render_info_card(
    "Итоговые метрики",
    [
        ("target_id", panel_summary.target_id),
        ("total_jobs", panel_summary.total_jobs),
        ("analyzed_jobs", panel_summary.analyzed_jobs),
        ("fully_scored_jobs", panel_summary.fully_scored_jobs),
        ("methods", ", ".join(panel_summary.methods)),
    ],
    accent="#5f3dc4",
)

display(top_consensus_df)
display(panel_rows_df.sort_values(['consensus_rank_desc', 'mutation'], na_position='last'))


## 8. Повторная сборка summary без нового запуска

Если основной расчёт уже был выполнен раньше, можно просто перечитать старый `state.sqlite` и заново собрать экспортные таблицы.


In [ ]:
existing_run_root = None
# existing_run_root = run_root

if existing_run_root:
    existing_run_root = Path(existing_run_root)
    existing_summary = summarize_panel_state_db(existing_run_root / "state.sqlite")
    existing_outputs = write_panel_summary_outputs(
        existing_summary,
        existing_run_root / summary_export_dirname,
    )
    render_info_card(
        "Summary rebuilt",
        [(key, value) for key, value in {k: str(v) for k, v in existing_outputs.items()}.items()],
        accent="#0b7285",
    )
    display(pd.DataFrame(existing_summary.top_consensus))
else:
    print("Если нужно просто пересобрать summary, укажи existing_run_root на старую директорию запуска.")
